In [ ]:
using Bessels, MPSDynamics, ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b_0 = 7.0515*2*pi

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.5127532
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
t_max = 95/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=1/(50*ω_a)
t_save = max(t_min_save,0):dt_save:t_max

re_a_0, im_a_0 = 15, -6
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
system_prepared = prepare(system)
environment = MeanField(κ_a, κ_b)
environment_prepared = prepare(system,environment)
u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Sol_vec = Vector{Tuple{Vector{Float64},Vector{Float64}}}()
Center_vec = Vector{Tuple{Float64,Float64}}()
Width_vec = Vector{Tuple{Float64,Float64}}()
ϵ_0_list = vcat(range(0.48,0.68,40))
for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10.12*g2
    drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)

    p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

    ϕ_a, _, q_a, _, _ = default_solver(u_dot_0, u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)
    Re = ϕ_a / sqrt(2*Z_a)
    Im = q_a * sqrt(Z_a/2)
    ω =  ω_a
    Re_2 =@. Re * cos(ω * t_list) + Im * sin(-ω * t_list)
    Im_2 =@. -Re * sin(-ω * t_list) + Im * cos(ω * t_list)
    push!(Sol_vec, (Re_2,Im_2))
    center = (sum(Re_2)/length(Re_2),sum(Im_2)/length(Im_2))
    push!(Center_vec, center)
    width = (maximum(Re_2)-minimum(Re_2),maximum(Im_2)-minimum(Im_2))
    push!(Width_vec, width)
end

In [ ]:
p0 = plot()
for (i,ϵ_0) in enumerate(ϵ_0_list)
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    scatter!(p0,Center_vec[i])
end
display(p0)

In [ ]:
scatter(abs.(getindex.(Center_vec,1)), abs.(getindex.(Center_vec,2)), zcolor=ϵ_0_list, ylims=(0,5))
x=range(start = 0,stop = 12, step = .1)
plot!(x,x.^2/30)

In [ ]:
scatter(getindex.(Width_vec,1), getindex.(Width_vec,2), zcolor=ϵ_0_list)

In [ ]:
ϵ_0_list[1]

In [ ]:
Test=[]
for ϵ_0 in ϵ_0_list
    α = -6/(0.625-0.48)
    β = 11-0.48*α
    re_a_0 = α*ϵ_0+β
    im_a_0 = re_a_0^2/30
    push!(Test,(re_a_0,im_a_0))
end

In [ ]:
Sol_vec = Vector{Tuple{Vector{Float64},Vector{Float64}}}()
Center_vec = Vector{Tuple{Float64,Float64}}()
Width_vec = Vector{Tuple{Float64,Float64}}()
ϵ_0_list = vcat(range(0.48,0.68,40))
for ϵ_0 in ϵ_0_list
    α = -6/(0.625-0.48)
    β = 11-0.48*α
    re_a_0 = α*ϵ_0+β
    im_a_0 = re_a_0^2/30
    ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
    ϕ_b_0, q_b_0=0.0, 0.0

    system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
    system_prepared = prepare(system)
    environment = MeanField(κ_a, κ_b)
    environment_prepared = prepare(system,environment)
    u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10.12*g2
    drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)

    p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

    ϕ_a, _, q_a, _, _ = default_solver(u_dot_0, u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)
    Re = ϕ_a / sqrt(2*Z_a)
    Im = q_a * sqrt(Z_a/2)
    ω =  ω_a
    Re_2 =@. Re * cos(ω * t_list) + Im * sin(-ω * t_list)
    Im_2 =@. -Re * sin(-ω * t_list) + Im * cos(ω * t_list)
    push!(Sol_vec, (Re_2,Im_2))
    center = (sum(Re_2)/length(Re_2),sum(Im_2)/length(Im_2))
    push!(Center_vec, center)
    width = (maximum(Re_2)-minimum(Re_2),maximum(Im_2)-minimum(Im_2))
    push!(Width_vec, width)
end

In [ ]:
Center_vec

In [ ]:
scatter(abs.(getindex.(Center_vec,1)), abs.(getindex.(Center_vec,2)), zcolor=ϵ_0_list, ylims=(0,5))
scatter!(abs.(getindex.(Test,1)), abs.(getindex.(Test,2)), zcolor=ϵ_0_list, ylims=(0,5))
x=range(start = 0,stop = 12, step = .1)
plot!(x,x.^2/30)